# 41 — JD Skill Extraction
**Goal:** Extract required and preferred skills from job descriptions.

Ch. 40 split the JD into sections; this chapter harvests the *skills* from those sections. A posting mentions a handful of technologies — "Python", "TensorFlow", "SQL" — and the extractor's job is to find every mention and decide whether the skill is **required** (lives in Qualifications) or **preferred** (lives in Nice to have). The distinction is not cosmetic: it changes how a candidate is scored.

**Why it matters for resumes / ATS:** skill matching is the heart of resume–JD comparison, and the *required vs preferred* flag is what makes a match meaningful. Missing a required skill should disqualify; missing a preferred skill should only cost a few points. An ATS that flattens this distinction either rejects good candidates (preferred treated as required) or lets weak ones through (required treated as preferred). This chapter is the JD-side twin of the skill extraction performed on resumes in earlier blocks.

## 1. Required vs Preferred Skills

The core idea: **a skill's section determines its weight**. A skill found in the *qualifications* bucket is required — a screening gate. The same skill in the *nice_to_have* bucket is preferred — a ranking signal. `extract_jd_skills()` checks each skill in a curated `SKILLS_DB` against each section's text and tags it accordingly.

**What the code does:**
- `SKILLS_DB` is the controlled vocabulary the extractor knows about (Python, TensorFlow, PyTorch, SQL, Spark, Docker, Kubernetes, AWS, NLP).
- It tries to reuse Ch. 40's `detect_jd_sections`; the guard is `'detect_jd_sections' in dir()` — but inside a function, `dir()` lists only *local* names, so this check is effectively always `False` and the fallback branch runs.
- The fallback places the whole JD text into both `qualifications` and `nice_to_have` buckets, so *every* found skill lands in both and is tagged **preferred**.

**Expected:** with the sample JD, `required` comes back empty and `preferred` contains `['Python', 'TensorFlow', 'PyTorch', 'SQL', 'NLP']` — all five skills the JD actually mentions (Spark/Docker/Kubernetes/AWS never appear). The intended design — skills from Qualifications → required, from Nice to have → preferred — would need `detect_jd_sections` passed in as an argument or the section logic inlined; as written, the section-aware path is dead code. The notebook also assumes Ch. 40's `jd` is still in the kernel, since the cell never redefines it.

In [ ]:
import re

SKILLS_DB = ["Python", "TensorFlow", "PyTorch", "SQL", "Spark", "Docker", "Kubernetes", "AWS", "NLP"]

def extract_jd_skills(jd_text, skills_db):
    required, preferred = [], []
    sections = detect_jd_sections(jd_text) if 'detect_jd_sections' in dir() else {"qualifications": [jd_text], "nice_to_have": [jd_text]}
    
    for skill in skills_db:
        found_in = []
        for sec_name, lines in sections.items():
            text = " ".join(lines).lower()
            if skill.lower() in text:
                found_in.append(sec_name)
        if "nice_to_have" in found_in:
            preferred.append(skill)
        elif found_in:
            required.append(skill)
    return required, preferred

req, pref = extract_jd_skills(jd, SKILLS_DB)
print("Required:")
for s in req: print(f"  - {s}")
print("\nPreferred:")
for s in pref: print(f"  - {s}")

## 2. Skill Frequency Analysis

Beyond *presence*, frequency is a cheap importance signal: a skill named three times is usually more central than one named once. `skill_frequency()` counts how often each vocabulary skill appears anywhere in the JD text and ranks skills by that count.

**What the code does:**
- Builds a word-level `Counter` with `re.findall(r"\b\w+\b", ...)` for general term frequencies.
- Then, per skill, counts *substring* occurrences with `jd_text.lower().count(skill.lower())` — simpler than the regex pass and case-insensitive, but it also counts partial matches ("Python" would match inside "Pythonic").
- Returns skills sorted by count descending; the cell prints the top 8.

**Expected:** in the sample JD every listed skill is mentioned exactly once, so the ranking shows `Python`, `TensorFlow`, `PyTorch`, `SQL`, `NLP` each at `mentioned 1x`. The output is flat here because the sample is short; on a real posting a skill repeated in both About and Qualifications would float to the top — which is exactly the signal the ranking is meant to expose. Note the print slice `[:8]` just truncates the list for display.

In [ ]:
from collections import Counter
import re

def skill_frequency(jd_text, skills_db):
    words = re.findall(r"\\b\\w+\\b", jd_text.lower())
    freq = Counter(words)
    # Check skills
    skill_counts = {}
    for skill in skills_db:
        count = jd_text.lower().count(skill.lower())
        if count > 0:
            skill_counts[skill] = count
    return sorted(skill_counts.items(), key=lambda x: -x[1])

for skill, count in skill_frequency(jd, SKILLS_DB)[:8]:
    print(f"  {skill:12s}: mentioned {count}x")

## Summary: Section-based extraction distinguishes required from preferred skills.

**Skill extraction is only as precise as the section boundaries feeding it — and the required/preferred split is a screening decision, not a display nicety.**

This chapter shows the two failure modes that plague real JD extractors: a vocabulary that silently drops unlisted skills (Spark, Docker, Kubernetes, AWS vanish because they are never mentioned — but also because the DB is finite), and a reuse guard that never fires, collapsing every skill into "preferred". The frequency pass adds a cheap importance signal on top. The extracted skills become the candidate set that Ch. 44 ranks and Ch. 46 matches against resume skills. Next, Ch. 42 extracts the *actions* a JD expects — responsibilities — using POS tagging.